# Task 2: Retrieval-Augmented Generation (RAG) on NIST AI RMF 1.0
**Tndrr AI Internship Assignment – Task 2**

### Objective
Build an end-to-end RAG system from scratch on the NIST AI Risk Management Framework 1.0 (48-page PDF document) without using LangChain or LlamaIndex.

### Core Architecture
- **Document:** NIST AI 100-1 (AI RMF 1.0), 48 pages, 15,400 words
- **Extraction:** Custom PyMuPDF extraction with text normalization and page tracking
- **Chunking Configurations:** 300-word vs 600-word sliding window (50-word overlap)
- **Embedding Model:** `sentence-transformers/all-MiniLM-L6-v2` (384-dimensional dense vectors)
- **Vector Store:** Pure NumPy cosine similarity matrix operations
- **Retrieval Metric:** Recall@5 on 15 manually verified gold questions
- **Generation:** Grounded prompt construction with citations `[chunk_id, page N]`

## 1. Environment Setup & Dependency Installation

In [ ]:
!pip install -q PyMuPDF sentence-transformers torch transformers numpy
import numpy as np
import fitz
import sentence_transformers
print('PyMuPDF version:', fitz.__version__)
print('SentenceTransformers:', sentence_transformers.__version__)
print('NumPy version:', np.__version__)


## 2. Clone Repository or Set Directory

In [ ]:
import os
from pathlib import Path

if not Path('download_document.py').exists():
    if Path('Tndrr-ai').exists():
        os.chdir('Tndrr-ai/tndrr-ai-internship-assignment/task2_rag')
    elif Path('tndrr-ai-internship-assignment/task2_rag').exists():
        os.chdir('tndrr-ai-internship-assignment/task2_rag')
    else:
        !git clone https://github.com/Rahul0Developer/Tndrr-ai.git
        os.chdir('Tndrr-ai/tndrr-ai-internship-assignment/task2_rag')

print('Working directory:', os.getcwd())
!ls -la


## 3. Extract and Clean Document Text with Page Tracking
Extracts 48 pages from `data/nist_ai_rmf_1.0.pdf` and applies cleaning rules.

In [ ]:
!python download_document.py
!python extract_pdf.py


## 4. Multi-Granularity Chunking (300w vs. 600w)
Generates two distinct chunking granularities with 50-word overlap across the continuous document stream.

In [ ]:
!python chunking.py


## 5. Dense Embedding Generation
Encodes chunks into L2-normalized 384-dimensional vectors using `all-MiniLM-L6-v2`.

In [ ]:
!python embeddings.py


## 6. Recall@5 Retrieval Benchmark on 15 Evaluation Questions
Calculates Recall@5 separately for both 300-word and 600-word configurations.

In [ ]:
!python evaluate_retrieval.py


## 7. Display Benchmark Results & Detailed Failure Diagnostics

In [ ]:
import json
from pathlib import Path

res_file = Path('results/recall_at_5.json')
if res_file.exists():
    data = json.loads(res_file.read_text(encoding='utf-8'))
    print('=' * 65)
    print('RETRIEVAL RECALL@5 BENCHMARK RESULTS')
    print('=' * 65)
    for cfg in ['c300', 'c600']:
        cinfo = data['configs'][cfg]
        label = '300-word chunks (c300)' if cfg == 'c300' else '600-word chunks (c600)'
        print(f"{label:<28}: Recall@5 = {cinfo['recall_at_5']:.1%} ({cinfo['hits']}/{cinfo['n_questions']} hits)")
    print('=' * 65)
    
    print('\nMISSED QUESTIONS (DIAGNOSTIC FAILURE CASES):')
    for cfg in ['c300', 'c600']:
        misses = [d for d in data['configs'][cfg]['details'] if not d['recalled']]
        print(f'\nConfiguration {cfg} Misses ({len(misses)} total):')
        for m in misses:
            top1 = m['top5'][0]
            print(f"  - [{m['id']}] {m['question']}")
            print(f"    Expected Page: {m['expected_page']} | Retrieved Top-1: {top1['chunk_id']} (Page {top1['page']}, Sim: {top1['score']:.3f})")
else:
    print('results/recall_at_5.json not found.')


## 8. Interactive End-to-End RAG Querying
Test live query retrieval with grounded context and source citations `[chunk_id, page N]`.

In [ ]:
# Grounded retrieval and prompt construction:
!python rag_pipeline.py --question "What are the four functions of the AI RMF Core?" --config c300 --dry-run

# Or with full Qwen generation model (if resources allow):
# !python rag_pipeline.py --question "What are the four functions of the AI RMF Core?" --config c300
